# Version B Full Experiment: Collaborative Filtering Line (B0-B5)

First run `DEBUG_MODE=True` to verify correctness.
Then set `DEBUG_MODE=False` and `FULL_RUN=True` for final results.

In [ ]:
import os
import json
import time
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

In [ ]:
DATA_DIR = "../../Data/Pure_Data"
RESULTS_DIR = "../Results"
FIGURES_DIR = "../Results/Figures"

DEBUG_MODE = False
MAX_DEBUG_USERS = 1000
FULL_RUN = True

POSITIVE_THRESHOLD = 4
K = 10
RANDOM_STATE = 42

BASE_M_VALUES = [5, 10, 20, 50, 100]
B1_K_VALUES = [10, 20, 50]
B2_K_VALUES = [10, 20, 50, 100]
B3_K_VALUES = [10, 20, 50, 100]
B4_COMPONENT_VALUES = [32, 64, 128]

if DEBUG_MODE:
    BASE_M_VALUES = [10, 50]
    B1_K_VALUES = [10, 20]
    B2_K_VALUES = [10, 20]
    B3_K_VALUES = [10, 20]
    B4_COMPONENT_VALUES = [32]

np.random.seed(RANDOM_STATE)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print("DEBUG_MODE:", DEBUG_MODE)
print("FULL_RUN:", FULL_RUN)
if (not DEBUG_MODE) and (not FULL_RUN):
    print("Warning: FULL_RUN=False while DEBUG_MODE=False. Proceeding, but this should be intentional.")

In [ ]:
required_files = [
    "interactions_filtered.csv",
    "interactions_train_filtered.csv",
    "interactions_test_filtered.csv",
    "recipe_model_table.csv",
]

if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(f"DATA_DIR not found: {DATA_DIR}")

for f in required_files:
    p = os.path.join(DATA_DIR, f)
    if not os.path.isfile(p):
        raise FileNotFoundError(f"Missing required file: {p}")

interactions_filtered = pd.read_csv(os.path.join(DATA_DIR, "interactions_filtered.csv"))
train = pd.read_csv(os.path.join(DATA_DIR, "interactions_train_filtered.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "interactions_test_filtered.csv"))
recipe_model_table = pd.read_csv(os.path.join(DATA_DIR, "recipe_model_table.csv"))

for name, df in [("train", train), ("test", test), ("interactions_filtered", interactions_filtered)]:
    for col in ["user_id", "recipe_id", "rating"]:
        if col not in df.columns:
            raise ValueError(f"{name} missing required column: {col}")

if train.empty or test.empty:
    raise ValueError("train or test is empty")

train_users = set(train["user_id"].unique())
test_users = set(test["user_id"].unique())
train_items = set(train["recipe_id"].unique())
test_items = set(test["recipe_id"].unique())

print("train shape:", train.shape)
print("test shape:", test.shape)
print("test users also in train:", len(test_users & train_users), " / ", len(test_users))
print("test recipes also in train:", len(test_items & train_items), " / ", len(test_items))
display(train.head())

In [ ]:
# Relevance definition alignment note:
# Version A protocol uses held-out test item setup. Version B explicitly uses positive test relevance.
# This notebook keeps that rule explicit and emits a comparison warning in config notes.

test_positive = test[test["rating"] >= POSITIVE_THRESHOLD].copy()

def build_user_sets(df):
    out = defaultdict(set)
    for u, i in zip(df["user_id"].values, df["recipe_id"].values):
        out[u].add(i)
    return out

user_train_all_items = build_user_sets(train)
user_train_positive_items = build_user_sets(train[train["rating"] >= POSITIVE_THRESHOLD])
user_test_relevant_items = build_user_sets(test_positive)

eval_users = sorted(user_test_relevant_items.keys())
if DEBUG_MODE:
    eval_users = eval_users[:MAX_DEBUG_USERS]
    user_test_relevant_items = {u: user_test_relevant_items[u] for u in eval_users}

print("Relevant test users for evaluation:", len(eval_users))
print("Positive threshold:", POSITIVE_THRESHOLD)

In [ ]:
def precision_at_k(rec, rel, k):
    rec = rec[:k]
    if k == 0 or len(rec) == 0:
        return 0.0
    hit = sum(1 for x in rec if x in rel)
    return hit / k

def recall_at_k(rec, rel, k):
    if len(rel) == 0:
        return 0.0
    rec = rec[:k]
    hit = sum(1 for x in rec if x in rel)
    return hit / len(rel)

def ndcg_at_k(rec, rel, k):
    rec = rec[:k]
    if len(rec) == 0 or len(rel) == 0:
        return 0.0
    dcg = 0.0
    for rank, item in enumerate(rec, start=1):
        if item in rel:
            dcg += 1.0 / np.log2(rank + 1)
    ideal = min(len(rel), k)
    idcg = sum(1.0 / np.log2(r + 1) for r in range(1, ideal + 1)) if ideal > 0 else 0.0
    return dcg / idcg if idcg > 0 else 0.0

def hit_at_k(rec, rel, k):
    rec = rec[:k]
    return 1.0 if any(x in rel for x in rec) else 0.0

def evaluate_model(model_id, model_name, phase, split_name, recommendations, runtime_seconds):
    per_user_rows = []
    unique_rec = set()
    total_recommendations = 0

    for u in eval_users:
        rel = user_test_relevant_items.get(u, set())
        rec_items = recommendations.get(u, [])
        rec_ids = [x[0] if isinstance(x, tuple) else x for x in rec_items][:K]
        for rid in rec_ids:
            unique_rec.add(rid)
        total_recommendations += len(rec_ids)

        p = precision_at_k(rec_ids, rel, K)
        r = recall_at_k(rec_ids, rel, K)
        n = ndcg_at_k(rec_ids, rel, K)
        h = hit_at_k(rec_ids, rel, K)

        per_user_rows.append({
            "user_id": u,
            "model_id": model_id,
            "precision_at_k": p,
            "recall_at_k": r,
            "ndcg_at_k": n,
            "hit_at_k": h,
            "num_relevant_items": len(rel),
            "num_recommended_items": len(rec_ids),
        })

    per_user_df = pd.DataFrame(per_user_rows)
    eval_count = len(per_user_df)
    catalog_coverage = len(unique_rec) / train["recipe_id"].nunique() if train["recipe_id"].nunique() > 0 else 0.0
    sec_per_user = runtime_seconds / eval_count if eval_count > 0 else np.nan

    agg = {
        "version": "B",
        "phase": phase,
        "model_id": model_id,
        "model_name": model_name,
        "split": split_name,
        "k": K,
        "evaluated_users": int(eval_count),
        "precision_at_k": float(per_user_df["precision_at_k"].mean()) if eval_count else 0.0,
        "recall_at_k": float(per_user_df["recall_at_k"].mean()) if eval_count else 0.0,
        "ndcg_at_k": float(per_user_df["ndcg_at_k"].mean()) if eval_count else 0.0,
        "hit_at_k": float(per_user_df["hit_at_k"].mean()) if eval_count else 0.0,
        "catalog_coverage_at_k": float(catalog_coverage),
        "total_recommendations": int(total_recommendations),
        "unique_recommended_items": int(len(unique_rec)),
        "runtime_seconds": float(runtime_seconds),
        "seconds_per_user": float(sec_per_user) if not np.isnan(sec_per_user) else np.nan,
    }
    return agg, per_user_df

In [ ]:
def popularity_recs_for_users(pop_ranked, users):
    out = {}
    for u in users:
        seen = user_train_all_items.get(u, set())
        recs = []
        for rid, score in pop_ranked:
            if rid in seen:
                continue
            recs.append((rid, float(score)))
            if len(recs) >= K:
                break
        out[u] = recs
    return out

def build_item_neighbor_cache(train_df, use_positive_only=True):
    if use_positive_only:
        work = train_df[train_df["rating"] >= POSITIVE_THRESHOLD].copy()
    else:
        work = train_df.copy()
    users = np.sort(work["user_id"].unique())
    items = np.sort(work["recipe_id"].unique())
    u2i = {u: idx for idx, u in enumerate(users)}
    r2i = {r: idx for idx, r in enumerate(items)}
    i2r = {idx: r for r, idx in r2i.items()}

    rows = work["user_id"].map(u2i).values
    cols = work["recipe_id"].map(r2i).values
    vals = np.ones(len(work), dtype=np.float32)
    ui = csr_matrix((vals, (rows, cols)), shape=(len(users), len(items)), dtype=np.float32)
    iu = ui.T.tocsr()
    return ui, iu, u2i, r2i, i2r

def user_knn_recommendations(ui_matrix, u2i, users_eval, k_neighbors, pop_ranked):
    nn = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=min(k_neighbors + 1, ui_matrix.shape[0]), n_jobs=-1)
    nn.fit(ui_matrix)
    dists, nbrs = nn.kneighbors(ui_matrix, return_distance=True)
    user_idx_to_id = {idx: uid for uid, idx in u2i.items()}
    recs = {}
    for u in users_eval:
        if u not in u2i:
            recs[u] = popularity_recs_for_users(pop_ranked, [u])[u]
            continue
        uidx = u2i[u]
        neigh_idxs = nbrs[uidx][1:]
        neigh_d = dists[uidx][1:]
        scores = defaultdict(float)
        seen = user_train_all_items.get(u, set())
        for nidx, d in zip(neigh_idxs, neigh_d):
            sim = 1.0 - float(d)
            if sim <= 0:
                continue
            nuid = user_idx_to_id.get(int(nidx))
            if nuid is None:
                continue
            for item in user_train_positive_items.get(nuid, set()):
                if item in seen:
                    continue
                scores[item] += sim
        if not scores:
            recs[u] = popularity_recs_for_users(pop_ranked, [u])[u]
        else:
            recs[u] = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:K]
    return recs

def item_knn_recommendations(iu_matrix, r2i, i2r, users_eval, k_neighbors, pop_ranked, weighted=False):
    nn = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=min(k_neighbors + 1, iu_matrix.shape[0]), n_jobs=-1)
    nn.fit(iu_matrix)
    dists, nbrs = nn.kneighbors(iu_matrix, return_distance=True)

    user_mean = train.groupby("user_id")["rating"].mean().to_dict() if weighted else {}
    recs = {}

    user_weight_map = defaultdict(dict)
    if weighted:
        for row in train.itertuples(index=False):
            w = float(getattr(row, "rating")) - float(user_mean.get(getattr(row, "user_id"), 0.0))
            user_weight_map[getattr(row, "user_id")][getattr(row, "recipe_id")] = w

    for u in users_eval:
        base_hist = user_train_positive_items.get(u, set())
        seen = user_train_all_items.get(u, set())
        if not base_hist:
            recs[u] = popularity_recs_for_users(pop_ranked, [u])[u]
            continue
        scores = defaultdict(float)
        for item in base_hist:
            idx = r2i.get(item)
            if idx is None:
                continue
            nbr_idx = nbrs[idx][1:]
            nbr_dist = dists[idx][1:]
            hist_w = 1.0
            if weighted:
                hist_w = user_weight_map.get(u, {}).get(item, 0.0)
                if hist_w == 0:
                    hist_w = 0.5
            for n_i, d in zip(nbr_idx, nbr_dist):
                sim = 1.0 - float(d)
                if sim <= 0:
                    continue
                cand = i2r[int(n_i)]
                if cand in seen:
                    continue
                scores[cand] += sim * hist_w
        if not scores:
            recs[u] = popularity_recs_for_users(pop_ranked, [u])[u]
        else:
            recs[u] = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:K]
    return recs

def svd_recommendations(train_df, users_eval, n_components, pop_ranked):
    users = np.sort(train_df["user_id"].unique())
    items = np.sort(train_df["recipe_id"].unique())
    u2i = {u: idx for idx, u in enumerate(users)}
    r2i = {r: idx for idx, r in enumerate(items)}
    i2r = {idx: r for r, idx in r2i.items()}

    rows = train_df["user_id"].map(u2i).values
    cols = train_df["recipe_id"].map(r2i).values
    vals = train_df["rating"].astype(float).values
    mat = csr_matrix((vals, (rows, cols)), shape=(len(users), len(items)), dtype=np.float32)

    n_comp = min(n_components, min(mat.shape) - 1) if min(mat.shape) > 1 else 1
    svd = TruncatedSVD(n_components=max(1, n_comp), random_state=RANDOM_STATE)
    user_emb = svd.fit_transform(mat)
    item_emb = svd.components_.T
    score_mat = user_emb @ item_emb.T

    recs = {}
    for u in users_eval:
        if u not in u2i:
            recs[u] = popularity_recs_for_users(pop_ranked, [u])[u]
            continue
        uidx = u2i[u]
        seen = user_train_all_items.get(u, set())
        s = score_mat[uidx]
        idx_rank = np.argsort(-s)
        out = []
        for idx in idx_rank:
            item = i2r[int(idx)]
            if item in seen:
                continue
            out.append((item, float(s[idx])))
            if len(out) >= K:
                break
        if not out:
            out = popularity_recs_for_users(pop_ranked, [u])[u]
        recs[u] = out
    return recs

In [ ]:
metrics_rows = []
tuning_rows = []
per_user_frames = []
runtime_rows = []

split_name = "filtered_temporal"

# ---- B0 Bayesian Popularity ----
b0_best = None
pop_global_mean = train["rating"].mean()
for m in tqdm(BASE_M_VALUES, desc="B0 tuning"):
    t0 = time.time()
    pop = train.groupby("recipe_id").agg(R=("rating", "mean"), v=("rating", "count")).reset_index()
    pop["score"] = (pop["v"] / (pop["v"] + m)) * pop["R"] + (m / (pop["v"] + m)) * pop_global_mean
    pop = pop.sort_values("score", ascending=False)
    pop_ranked = list(zip(pop["recipe_id"].tolist(), pop["score"].tolist()))
    recs = popularity_recs_for_users(pop_ranked, eval_users)
    runtime = time.time() - t0
    agg, per_user = evaluate_model(f"B0_m{m}", "Bayesian popularity baseline", "B0", split_name, recs, runtime)
    agg["parameters"] = json.dumps({"m": m})
    agg["notes"] = "Bayesian popularity smoothing"
    tuning_rows.append(agg.copy())
    if (b0_best is None) or (agg["ndcg_at_k"] > b0_best["agg"]["ndcg_at_k"]):
        b0_best = {"agg": agg, "per_user": per_user, "ranked": pop_ranked, "m": m}

metrics_rows.append(b0_best["agg"])
per_user_frames.append(b0_best["per_user"])
runtime_rows.append({
    "phase": "B0",
    "model_id": b0_best["agg"]["model_id"],
    "runtime_seconds": b0_best["agg"]["runtime_seconds"],
    "evaluated_users": b0_best["agg"]["evaluated_users"],
    "seconds_per_user": b0_best["agg"]["seconds_per_user"],
})
pop_ranked_best = b0_best["ranked"]

# ---- B1 user-kNN ----
ui, iu, u2i, r2i, i2r = build_item_neighbor_cache(train, use_positive_only=True)
b1_best = None
for kk in tqdm(B1_K_VALUES, desc="B1 tuning"):
    t0 = time.time()
    recs = user_knn_recommendations(ui, u2i, eval_users, kk, pop_ranked_best)
    runtime = time.time() - t0
    agg, per_user = evaluate_model(f"B1_k{kk}", "User-based kNN CF", "B1", split_name, recs, runtime)
    agg["parameters"] = json.dumps({"k_neighbors": kk, "metric": "cosine"})
    agg["notes"] = "Fallback to B0 for insufficient user history"
    tuning_rows.append(agg.copy())
    if (b1_best is None) or (agg["ndcg_at_k"] > b1_best["agg"]["ndcg_at_k"]):
        b1_best = {"agg": agg, "per_user": per_user, "recs": recs}
metrics_rows.append(b1_best["agg"])
per_user_frames.append(b1_best["per_user"])
runtime_rows.append({"phase": "B1", "model_id": b1_best["agg"]["model_id"], "runtime_seconds": b1_best["agg"]["runtime_seconds"], "evaluated_users": b1_best["agg"]["evaluated_users"], "seconds_per_user": b1_best["agg"]["seconds_per_user"]})

# ---- B2 item-kNN implicit ----
b2_best = None
for kk in tqdm(B2_K_VALUES, desc="B2 tuning"):
    t0 = time.time()
    recs = item_knn_recommendations(iu, r2i, i2r, eval_users, kk, pop_ranked_best, weighted=False)
    runtime = time.time() - t0
    agg, per_user = evaluate_model(f"B2_k{kk}", "Item-based kNN CF implicit", "B2", split_name, recs, runtime)
    agg["parameters"] = json.dumps({"k_neighbors": kk, "metric": "cosine", "positive_threshold": POSITIVE_THRESHOLD})
    agg["notes"] = "Implicit positive-only item-kNN"
    tuning_rows.append(agg.copy())
    if (b2_best is None) or (agg["ndcg_at_k"] > b2_best["agg"]["ndcg_at_k"]):
        b2_best = {"agg": agg, "per_user": per_user, "recs": recs}
metrics_rows.append(b2_best["agg"])
per_user_frames.append(b2_best["per_user"])
runtime_rows.append({"phase": "B2", "model_id": b2_best["agg"]["model_id"], "runtime_seconds": b2_best["agg"]["runtime_seconds"], "evaluated_users": b2_best["agg"]["evaluated_users"], "seconds_per_user": b2_best["agg"]["seconds_per_user"]})

# ---- B3 item-kNN weighted ----
b3_best = None
for kk in tqdm(B3_K_VALUES, desc="B3 tuning"):
    t0 = time.time()
    recs = item_knn_recommendations(iu, r2i, i2r, eval_users, kk, pop_ranked_best, weighted=True)
    runtime = time.time() - t0
    agg, per_user = evaluate_model(f"B3_k{kk}", "Item-based kNN CF rating-weighted", "B3", split_name, recs, runtime)
    agg["parameters"] = json.dumps({"k_neighbors": kk, "weighting": "rating_minus_user_mean"})
    agg["notes"] = "Weighted item-kNN using centered user ratings"
    tuning_rows.append(agg.copy())
    if (b3_best is None) or (agg["ndcg_at_k"] > b3_best["agg"]["ndcg_at_k"]):
        b3_best = {"agg": agg, "per_user": per_user, "recs": recs}
metrics_rows.append(b3_best["agg"])
per_user_frames.append(b3_best["per_user"])
runtime_rows.append({"phase": "B3", "model_id": b3_best["agg"]["model_id"], "runtime_seconds": b3_best["agg"]["runtime_seconds"], "evaluated_users": b3_best["agg"]["evaluated_users"], "seconds_per_user": b3_best["agg"]["seconds_per_user"]})

# ---- B4 SVD ----
b4_best = None
for comp in tqdm(B4_COMPONENT_VALUES, desc="B4 tuning"):
    t0 = time.time()
    recs = svd_recommendations(train, eval_users, comp, pop_ranked_best)
    runtime = time.time() - t0
    agg, per_user = evaluate_model(f"B4_svd{comp}", f"SVD CF {comp}", "B4", split_name, recs, runtime)
    agg["parameters"] = json.dumps({"n_components": comp})
    agg["notes"] = "TruncatedSVD collaborative latent factors"
    tuning_rows.append(agg.copy())
    if (b4_best is None) or (agg["ndcg_at_k"] > b4_best["agg"]["ndcg_at_k"]):
        b4_best = {"agg": agg, "per_user": per_user, "recs": recs}
metrics_rows.append(b4_best["agg"])
per_user_frames.append(b4_best["per_user"])
runtime_rows.append({"phase": "B4", "model_id": b4_best["agg"]["model_id"], "runtime_seconds": b4_best["agg"]["runtime_seconds"], "evaluated_users": b4_best["agg"]["evaluated_users"], "seconds_per_user": b4_best["agg"]["seconds_per_user"]})

# ---- B5 final comparison ----
metrics_df = pd.DataFrame(metrics_rows)
tuning_df = pd.DataFrame(tuning_rows)
per_user_df = pd.concat(per_user_frames, ignore_index=True) if per_user_frames else pd.DataFrame()
runtime_df = pd.DataFrame(runtime_rows)

best_final = metrics_df.sort_values(["ndcg_at_k", "recall_at_k", "precision_at_k"], ascending=False).iloc[0]
final_model_id = best_final["model_id"]

if final_model_id.startswith("B0"):
    final_recs = popularity_recs_for_users(pop_ranked_best, eval_users)
elif final_model_id.startswith("B1"):
    final_recs = b1_best["recs"]
elif final_model_id.startswith("B2"):
    final_recs = b2_best["recs"]
elif final_model_id.startswith("B3"):
    final_recs = b3_best["recs"]
else:
    final_recs = b4_best["recs"]

display(metrics_df[["model_id","model_name","precision_at_k","recall_at_k","ndcg_at_k","catalog_coverage_at_k","runtime_seconds"]])
print("Selected final model_id:", final_model_id)

In [ ]:
prefix = "debug_" if DEBUG_MODE else ""

metrics_path = os.path.join(RESULTS_DIR, f"{prefix}version_b_metrics.csv")
tuning_path = os.path.join(RESULTS_DIR, f"{prefix}version_b_tuning_results.csv")
per_user_path = os.path.join(RESULTS_DIR, f"{prefix}version_b_per_user_metrics.csv")
runtime_path = os.path.join(RESULTS_DIR, f"{prefix}version_b_phase_runtime.csv")
config_path = os.path.join(RESULTS_DIR, f"{prefix}version_b_config.json")
notes_path = os.path.join(RESULTS_DIR, f"{prefix}version_b_model_notes.md")
examples_path = os.path.join(RESULTS_DIR, f"{prefix}version_b_example_recommendations.csv")
top10_path = os.path.join(RESULTS_DIR, f"{prefix}version_b_top10_recommendations.csv")

metrics_df.to_csv(metrics_path, index=False)
tuning_df.to_csv(tuning_path, index=False)
per_user_df.to_csv(per_user_path, index=False)
runtime_df.to_csv(runtime_path, index=False)

config_payload = {
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "data_files": required_files,
    "relevant_item_definition": f"rating >= {POSITIVE_THRESHOLD}",
    "split": split_name,
    "k": K,
    "b0_m_values": BASE_M_VALUES,
    "b1_k_values": B1_K_VALUES,
    "b2_k_values": B2_K_VALUES,
    "b3_k_values": B3_K_VALUES,
    "b4_component_values": B4_COMPONENT_VALUES,
    "debug_mode": DEBUG_MODE,
    "full_run": FULL_RUN,
    "comparison_warning": "Version A/B evaluated_users may differ if relevance definition differs."
}
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config_payload, f, indent=2)

with open(notes_path, "w", encoding="utf-8") as f:
    f.write("# Version B Model Notes\n\n")
    for _, row in metrics_df.iterrows():
        f.write(f"## {row['model_id']} - {row['model_name']}\n")
        f.write(f"- phase: {row['phase']}\n")
        f.write(f"- precision@{K}: {row['precision_at_k']:.6f}\n")
        f.write(f"- recall@{K}: {row['recall_at_k']:.6f}\n")
        f.write(f"- ndcg@{K}: {row['ndcg_at_k']:.6f}\n")
        f.write(f"- hit@{K}: {row['hit_at_k']:.6f}\n")
        f.write(f"- coverage@{K}: {row['catalog_coverage_at_k']:.6f}\n")
        f.write(f"- runtime_seconds: {row['runtime_seconds']:.4f}\n")
        f.write(f"- parameters: {row.get('parameters','{}')}\n")
        f.write(f"- notes: {row.get('notes','')}\n\n")

example_users = eval_users[:10]
example_rows = []
for u in example_users:
    hist = sorted(list(user_train_all_items.get(u, set())))
    rel = sorted(list(user_test_relevant_items.get(u, set())))
    recs = final_recs.get(u, [])
    rec_ids = [x[0] if isinstance(x, tuple) else x for x in recs]
    example_rows.append({
        "user_id": u,
        "model_id": final_model_id,
        "user_history_items": "|".join(map(str, hist[:50])),
        "recommended_items": "|".join(map(str, rec_ids)),
        "relevant_test_items": "|".join(map(str, rel)),
        "explanation": "Top-N recommendation from selected collaborative model with train-history filtering.",
    })
pd.DataFrame(example_rows).to_csv(examples_path, index=False)

top_rows = []
for u in eval_users:
    recs = final_recs.get(u, [])
    for rk, rs in enumerate(recs[:K], start=1):
        rid, score = rs if isinstance(rs, tuple) else (rs, np.nan)
        top_rows.append({
            "user_id": u,
            "rank": rk,
            "recipe_id": rid,
            "score": float(score) if pd.notna(score) else np.nan,
            "model_id": final_model_id,
            "model_name": metrics_df.loc[metrics_df["model_id"] == final_model_id, "model_name"].iloc[0],
        })
pd.DataFrame(top_rows).to_csv(top10_path, index=False)

print(metrics_path)
print(tuning_path)
print(per_user_path)
print(runtime_path)
print(config_path)
print(notes_path)
print(examples_path)
print(top10_path)

In [ ]:
plot_df = metrics_df.copy()
plot_df = plot_df.sort_values("model_id")

plt.figure(figsize=(10,5))
x = np.arange(len(plot_df))
w = 0.25
plt.bar(x-w, plot_df["precision_at_k"], w, label="precision_at_k")
plt.bar(x, plot_df["recall_at_k"], w, label="recall_at_k")
plt.bar(x+w, plot_df["ndcg_at_k"], w, label="ndcg_at_k")
plt.xticks(x, plot_df["model_id"], rotation=20)
plt.title("Version B Model Metrics at 10")
plt.xlabel("model_id")
plt.ylabel("metric value")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, f"{prefix}version_b_model_metrics_at_10.png"), dpi=150)
plt.close()

plt.figure(figsize=(8,4.5))
plt.bar(plot_df["model_id"], plot_df["hit_at_k"])
plt.title("Version B Hit Rate at 10")
plt.xlabel("model_id")
plt.ylabel("hit_at_k")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, f"{prefix}version_b_hit_rate_at_10.png"), dpi=150)
plt.close()

plt.figure(figsize=(8,4.5))
plt.bar(plot_df["model_id"], plot_df["catalog_coverage_at_k"])
plt.title("Version B Catalog Coverage at 10")
plt.xlabel("model_id")
plt.ylabel("catalog_coverage_at_k")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, f"{prefix}version_b_catalog_coverage_at_10.png"), dpi=150)
plt.close()

plt.figure(figsize=(8,5))
plt.scatter(plot_df["seconds_per_user"], plot_df["ndcg_at_k"])
for _, r in plot_df.iterrows():
    plt.annotate(r["model_id"], (r["seconds_per_user"], r["ndcg_at_k"]), xytext=(4,4), textcoords="offset points", fontsize=8)
plt.title("Version B Quality vs Runtime at 10")
plt.xlabel("seconds_per_user")
plt.ylabel("ndcg_at_k")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, f"{prefix}version_b_quality_vs_runtime_at_10.png"), dpi=150)
plt.close()

plt.figure(figsize=(8,4.5))
rt = runtime_df.sort_values("phase")
plt.bar(rt["phase"], rt["runtime_seconds"])
plt.title("Version B Phase Runtime")
plt.xlabel("phase")
plt.ylabel("runtime_seconds")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, f"{prefix}version_b_phase_runtime.png"), dpi=150)
plt.close()

for prefix_phase, fname in [("B1_", "version_b_knn_tuning_curves.png"), ("B4_", "version_b_svd_tuning_curves.png")]:
    pass

knn_tune = tuning_df[tuning_df["model_id"].str.startswith(("B1_","B2_","B3_"), na=False)].copy()
if not knn_tune.empty:
    knn_tune["k_neighbors"] = knn_tune["model_id"].str.extract(r'k(\\d+)').astype(float)
    plt.figure(figsize=(9,5))
    for g, grp in knn_tune.groupby("phase"):
        grp = grp.sort_values("k_neighbors")
        plt.plot(grp["k_neighbors"], grp["ndcg_at_k"], marker="o", label=f"{g} ndcg")
    plt.title("Version B kNN Tuning Curves")
    plt.xlabel("K_neighbors")
    plt.ylabel("ndcg_at_k")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, f"{prefix}version_b_knn_tuning_curves.png"), dpi=150)
    plt.close()

svd_tune = tuning_df[tuning_df["model_id"].str.startswith("B4_", na=False)].copy()
if not svd_tune.empty:
    svd_tune["n_components"] = svd_tune["model_id"].str.extract(r'svd(\\d+)').astype(float)
    svd_tune = svd_tune.sort_values("n_components")
    plt.figure(figsize=(8,5))
    plt.plot(svd_tune["n_components"], svd_tune["precision_at_k"], marker="o", label="precision")
    plt.plot(svd_tune["n_components"], svd_tune["recall_at_k"], marker="o", label="recall")
    plt.plot(svd_tune["n_components"], svd_tune["ndcg_at_k"], marker="o", label="ndcg")
    plt.title("Version B SVD Tuning Curves")
    plt.xlabel("n_components")
    plt.ylabel("metric value")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, f"{prefix}version_b_svd_tuning_curves.png"), dpi=150)
    plt.close()

heat_cols = ["precision_at_k","recall_at_k","ndcg_at_k","hit_at_k","catalog_coverage_at_k","seconds_per_user"]
heat = plot_df.set_index("model_id")[heat_cols]
plt.figure(figsize=(9,4.5))
plt.imshow(heat.values, aspect="auto")
plt.xticks(range(len(heat_cols)), heat_cols, rotation=30, ha="right")
plt.yticks(range(len(heat.index)), heat.index)
plt.colorbar(label="value")
plt.title("Version B Model Comparison Heatmap")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, f"{prefix}version_b_model_comparison_heatmap.png"), dpi=150)
plt.close()

if not per_user_df.empty:
    box_models = sorted(per_user_df["model_id"].unique())
    box_data = [per_user_df.loc[per_user_df["model_id"] == m, "ndcg_at_k"].values for m in box_models]
    plt.figure(figsize=(10,5))
    plt.boxplot(box_data, labels=box_models, showfliers=False)
    plt.xticks(rotation=25)
    plt.title("Version B Per-user NDCG@10 Distribution")
    plt.ylabel("ndcg_at_k")
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, f"{prefix}version_b_per_user_metric_boxplot.png"), dpi=150)
    plt.close()

rank_hits = []
for u in eval_users:
    rel = user_test_relevant_items.get(u, set())
    recs = final_recs.get(u, [])
    rec_ids = [x[0] if isinstance(x, tuple) else x for x in recs][:K]
    for rnk, rid in enumerate(rec_ids, start=1):
        if rid in rel:
            rank_hits.append(rnk)
            break
if rank_hits:
    plt.figure(figsize=(7,4.5))
    bins = np.arange(1, K + 2) - 0.5
    plt.hist(rank_hits, bins=bins, rwidth=0.9)
    plt.xticks(range(1, K+1))
    plt.title("Version B Hit Rank Distribution at 10")
    plt.xlabel("hit rank")
    plt.ylabel("count")
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, f"{prefix}version_b_hit_rank_distribution_at_10.png"), dpi=150)
    plt.close()

top10_df = pd.read_csv(top10_path)
if "score" in top10_df.columns and top10_df["score"].notna().any():
    uid = top10_df["user_id"].iloc[0]
    sample = top10_df[top10_df["user_id"] == uid].sort_values("rank").head(10)
    plt.figure(figsize=(8,4.5))
    plt.bar(sample["rank"].astype(str), sample["score"].astype(float))
    plt.title(f"Version B Sample Recommendation Scores (user={uid})")
    plt.xlabel("rank")
    plt.ylabel("score")
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, f"{prefix}version_b_sample_recommendation_scores.png"), dpi=150)
    plt.close()

num_users = interactions_filtered["user_id"].nunique()
num_items = interactions_filtered["recipe_id"].nunique()
num_int = len(interactions_filtered)
density = num_int / (num_users * num_items)
sparsity = 1 - density
ctx_labels = ["users","recipes","interactions","density","sparsity"]
ctx_vals = [num_users, num_items, num_int, density, sparsity]
plt.figure(figsize=(8,4.5))
plt.bar(ctx_labels, ctx_vals)
plt.title("Version B Data Sparsity Context")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, f"{prefix}version_b_data_sparsity_context.png"), dpi=150)
plt.close()

print("Figures generated in:", FIGURES_DIR)